In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Raghav\\Desktop\\Image_Captioning_using_DeepLearning\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Raghav\\Desktop\\Image_Captioning_using_DeepLearning'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    images_dir: Path
    captions_file: Path
    train_img_id_path: Path
    val_img_id_path: Path
    test_img_id_path: Path
    train_imagesid_captions_path: Path
    val_imagesid_captions_path: Path
    test_imagesid_captions_path: Path

    TRAIN_SPLIT: float
    TEST_SPLIT: float
    RANDOM_STATE: int


In [6]:
from imgCaption.constants import *
from imgCaption.utils.common import read_yaml,create_directories

In [7]:
class ConfigurationManager:
    def __init__(self,config_filepath=CONFIG_FILE_PATH,params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:

        config = self.config.data_transformation
        params = self.params.data_transformation_params

        data_transformation_config = DataTransformationConfig(root_dir=config.root_dir,
                                                              images_dir=config.images_dir,
                                                              captions_file=config.captions_file,
                                                              train_img_id_path=config.train_img_id_path,
                                                              val_img_id_path=config.val_img_id_path,
                                                              test_img_id_path=config.test_img_id_path,
                                                              train_imagesid_captions_path=config.train_imagesid_captions_path,
                                                              val_imagesid_captions_path=config.val_imagesid_captions_path,
                                                              test_imagesid_captions_path=config.test_imagesid_captions_path,
                                                              TRAIN_SPLIT=params.TRAIN_SPLIT,
                                                              TEST_SPLIT=params.TEST_SPLIT,
                                                              RANDOM_STATE=params.RANDOM_STATE)
        
        return data_transformation_config

In [8]:
import os
import re
import json
from imgCaption import logger
from sklearn.model_selection import train_test_split

In [9]:
class DataTransformation:
    def __init__(self,config : DataTransformationConfig):
        self.config = config

    def split_images_ID(self):
        images_Id = os.listdir(self.config.images_dir)
        train_image_Id, val_test_Id = train_test_split(images_Id, test_size=self.config.TRAIN_SPLIT, random_state=self.config.RANDOM_STATE)
        val_image_Id, test_image_Id = train_test_split(val_test_Id, test_size=self.config.TEST_SPLIT, random_state=self.config.RANDOM_STATE)

        logger.info(f"Total images     : {len(images_Id)}")
        logger.info(f"Train images     : {len(train_image_Id)}")
        logger.info(f"Validation images: {len(val_image_Id)}")
        logger.info(f"Test images      : {len(test_image_Id)}")

        os.makedirs(self.config.root_dir, exist_ok=True)

        with open(self.config.train_img_id_path,"w") as f:
            for img_id in train_image_Id:
                f.write(f"{img_id}\n")

        logger.info("train_img_id_path created")

        with open(self.config.val_img_id_path,"w") as f:
            for img_id in val_image_Id:
                f.write(f"{img_id}\n")

        logger.info("val_img_id_path created")

        with open(self.config.test_img_id_path,"w") as f:
            for img_id in test_image_Id:
                f.write(f"{img_id}\n")

        logger.info("test_img_id_path created")

    def cleaned_captions(self):
        with open(self.config.captions_file) as f:
            full_captions = f.readlines()

        captions = [cap.lower().split(',')[1] for cap in full_captions[1:]]

        cleaned_captions = [
        re.sub(r'\s+', ' ',          
        re.sub(r'\d+', '',            
        re.sub(r'[^\w\s]', '', cap)   
        )).strip().lower()            
        for cap in captions
        ]

        images_ID = [ID.split(',')[0] for ID in full_captions[1:]]

        full_cleaned_captions = []

        for img_id, caption in zip(images_ID, cleaned_captions):
            formatted_str = f"{img_id}\tstartseq {caption} endseq"
            full_cleaned_captions.append(formatted_str)
    
    
        logger.info("captions cleaned and tokenized successfully")

        return full_cleaned_captions
    
    def create_mapping_dict(self, cleaned_captions: list, img_ids_path: Path, data_path: Path):
       
        with open(img_ids_path, "r") as f:
            target_ids = set(line.strip() for line in f.readlines())

        map = {}

        for entry in cleaned_captions:

            img_id,caption = entry.split('\t')

            if img_id in target_ids:
                if img_id not in map:
                    map[img_id] = []

                map[img_id].append(caption)

        logger.info(f"Successfully made {Path(img_ids_path).name} a dictionary")

        with open(data_path, 'w') as f:
            json.dump(map, f, indent=2)

        logger.info(f"File : {Path(data_path).name} has saved successfully")


In [11]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.split_images_ID()
    cleaned_captions = data_transformation.cleaned_captions()

    splits = [
        (data_transformation_config.train_img_id_path, data_transformation_config.train_imagesid_captions_path),
        (data_transformation_config.val_img_id_path,   data_transformation_config.val_imagesid_captions_path),
        (data_transformation_config.test_img_id_path,  data_transformation_config.test_imagesid_captions_path),
    ]

    for img_ids_path, data_path in splits:
        data_transformation.create_mapping_dict(cleaned_captions=cleaned_captions,
                                                img_ids_path=img_ids_path,
                                                data_path=data_path)
except Exception as e:
    raise e

[2026-06-26 11:27:52,427: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-26 11:27:52,459: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-26 11:27:52,463: INFO: common: created directory at: artifacts]
[2026-06-26 11:27:52,490: INFO: 272020008: Total images     : 8091]
[2026-06-26 11:27:52,492: INFO: 272020008: Train images     : 5663]
[2026-06-26 11:27:52,495: INFO: 272020008: Validation images: 1214]
[2026-06-26 11:27:52,495: INFO: 272020008: Test images      : 1214]
[2026-06-26 11:27:52,507: INFO: 272020008: train_img_id_path created]
[2026-06-26 11:27:52,517: INFO: 272020008: val_img_id_path created]
[2026-06-26 11:27:52,517: INFO: 272020008: test_img_id_path created]
[2026-06-26 11:27:53,273: INFO: 272020008: captions cleaned and tokenized successfully]
[2026-06-26 11:27:53,379: INFO: 272020008: Successfully made train_ids.txt a dictionary]
[2026-06-26 11:27:53,490: INFO: 272020008: File : train_imagesid_captions.json has saved success